# Estimativa de Movimento de Câmera FLIR — Rastreamento de UAP

## Referência visual: características do vídeo FLIR de sistema de armas

Baseado na análise visual de frames típicos desse tipo de vídeo, o código lida com:

| Elemento | Aparência | Tratamento |
|---|---|---|
| **Áreas pretas irregulares** | Blocos pretos nos cantos/bordas — overlay do sistema, fora do campo da câmera | Mascaradas automaticamente por limiar de intensidade + morfologia |
| **Alvo UAP** | Blob branco intenso (white-hot) ou escuro (black-hot) próximo ao centro | Detectado adaptativamente por percentil e mascarado com dilatação |
| **Mira / crosshair** | Símbolo + no centro da imagem, fixo | Banda horizontal + vertical + caixa central mascaradas |
| **Texto HUD** | Letras como 'N' (bússola), números de altitude, etc. | Mascaráveis via `hud_boxes` ou por detecção de texto branco |
| **Background** | Nuvens / terreno com textura moderada | Único elemento usado para estimativa de movimento |

## Abordagem

```
Frame FLIR
   │
   ├─► _find_valid_region()     detecta área não-preta (campo real da câmera)
   ├─► _detect_polarity()       identifica white-hot vs black-hot automaticamente
   ├─► _mask_target_blob()      remove UAP (blob de alto contraste)
   ├─► _mask_reticle()          remove mira/crosshair central
   └─► máscara final de background
           │
           ▼
   Shi-Tomasi corners → Lucas-Kanade pyramidal → vetores de deslocamento
           │
           ▼
   Filtragem MAD (rejeita outliers) → mediana robusta → (bg_dx, bg_dy)
           │
           ▼
   Pré-filtro mediana móvel → soma cumulativa → posição câmera
           │
           ▼
   Savitzky-Golay (deriv=0,1,2) → posição suave, velocidade, aceleração
```

## Convenção de sinal
- `cam_pos_x > 0` → câmera girou para a **direita** (pan direita)  
- `cam_pos_y > 0` → câmera inclinou para **baixo** (coords de imagem: y cresce para baixo)  
- `bg_dx_px` = deslocamento do **fundo** (sinal inverso à câmera)

## Dependências

```bash
pip install opencv-python numpy pandas matplotlib scipy
```

In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

try:
    from scipy.signal import savgol_filter
    _SCIPY = True
except ImportError:
    _SCIPY = False
    warnings.warn("scipy não encontrado — instale com: pip install scipy\n"
                  "Usando numpy.gradient como fallback (resultado mais ruidoso).")

try:
    import cv2
except ImportError as _err:
    raise ImportError("OpenCV é necessário: pip install opencv-python") from _err

print(f"OK  OpenCV {cv2.__version__}")
print(f"OK  scipy={'disponível' if _SCIPY else 'AUSENTE — instale para melhores resultados'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 1 — PRÉ-PROCESSAMENTO E MÁSCARA AUTOMÁTICA
# ═══════════════════════════════════════════════════════════════════════════════

def _to_gray(frame: np.ndarray) -> np.ndarray:
    """
    Converte frame BGR (ou já cinza) para uint8 single-channel.
    Aplica blur 3×3 suave para reduzir ruído de compressão de vídeo
    sem destruir as bordas de textura usadas no rastreamento.
    """
    if frame.ndim == 3:
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        g = frame.copy()
    return cv2.GaussianBlur(g, (3, 3), 0)


def _find_valid_region(
    gray: np.ndarray,
    black_thresh: int = 18,
    close_px: int = 25,
    erode_px: int = 10,
) -> np.ndarray:
    """
    Detecta a região visível (não-preta) do frame FLIR.

    Em vídeos de targeting pod e câmeras de sistemas de armas, o campo de visão
    da câmera NÃO preenche o frame inteiro. As bordas/cantos são preenchidas com
    blocos pretos irregulares pelo pipeline de renderização da HUD. Esses blocos
    têm intensidade ≈ 0 e precisam ser removidos ANTES de estimar o movimento.

    Algoritmo:
      1. Limiar simples: pixels > black_thresh são candidatos a válidos
      2. Fechamento morfológico: une regiões vizinhas separadas por costuras
      3. Erosão: afasta a máscara das bordas dos blocos pretos (evita artefatos)

    Parâmetros
    ----------
    gray        : frame em escala de cinza (uint8)
    black_thresh: pixels abaixo desse valor são considerados pretos/overlay
    close_px    : tamanho do kernel de fechamento (preenche lacunas internas)
    erode_px    : erosão final para distância segura das bordas pretas

    Retorna
    -------
    Máscara uint8: 255 = área válida, 0 = overlay preto
    """
    # Passo 1: todos os pixels acima do limiar são candidatos
    valid = (gray > black_thresh).astype(np.uint8) * 255

    # Passo 2: fechar lacunas entre tiles adjacentes da HUD
    k_close = cv2.getStructuringElement(cv2.MORPH_RECT, (close_px, close_px))
    valid = cv2.morphologyEx(valid, cv2.MORPH_CLOSE, k_close)

    # Passo 3: recuar da borda preta para evitar pixels de transição
    k_erode = cv2.getStructuringElement(cv2.MORPH_RECT, (erode_px, erode_px))
    valid = cv2.erode(valid, k_erode)

    return valid


def _detect_thermal_polarity(
    gray: np.ndarray,
    valid_region: np.ndarray,
    center_frac: float = 0.15,
) -> str:
    """
    Detecta automaticamente se a câmera FLIR está em modo white-hot ou black-hot.

    Em modo WHITE-HOT: objetos quentes aparecem como pixels claros (alto valor).
    Em modo BLACK-HOT: objetos quentes aparecem como pixels escuros (baixo valor).

    Heurística: compara a intensidade média da região central (onde o alvo UAP
    rastreado pelo sistema de armas tende a ficar) com a média do resto do frame.
    Se o centro for mais brilhante que o fundo → white-hot; caso contrário → black-hot.

    Retorna 'white_hot' ou 'black_hot'
    """
    h, w = gray.shape[:2]
    cy, cx = h // 2, w // 2
    ch = max(1, int(h * center_frac / 2))
    cw = max(1, int(w * center_frac / 2))

    center_mask = np.zeros((h, w), dtype=np.uint8)
    center_mask[cy - ch : cy + ch, cx - cw : cx + cw] = 255
    center_mask = cv2.bitwise_and(center_mask, valid_region)

    surround_mask = cv2.bitwise_and(
        valid_region,
        cv2.bitwise_not(center_mask),
    )

    center_px = gray[center_mask > 0]
    surround_px = gray[surround_mask > 0]

    if center_px.size < 10 or surround_px.size < 10:
        return "white_hot"  # default seguro

    return "white_hot" if float(np.mean(center_px)) >= float(np.mean(surround_px)) else "black_hot"


def _mask_target_blob(
    gray: np.ndarray,
    valid_region: np.ndarray,
    polarity: str = "white_hot",
    percentile: float = 99.0,
    dilation_px: int = 28,
    min_blob_px: int = 4,
) -> np.ndarray:
    """
    Detecta e mascara o blob de alto contraste (o alvo UAP) na região válida.

    O alvo é o elemento de MAIOR contraste relativo ao background. Em white-hot
    é o mais brilhante; em black-hot é o mais escuro. A detecção usa limiar
    adaptativo baseado no percentil da distribuição de intensidade da região
    válida, garantindo que o limiar se ajuste ao brilho médio do frame.

    Após detectar o blob principal, dilata a máscara para cobrir o halo
    térmico ao redor do alvo (brilho espalhado por difusão óptica e compressão).

    Parâmetros
    ----------
    polarity    : 'white_hot' ou 'black_hot'
    percentile  : limiar percentil (99 = mascara o 1% mais extremo)
    dilation_px : raio de dilatação em pixels ao redor do blob detectado
    min_blob_px : área mínima em pixels para considerar um blob

    Retorna
    -------
    Máscara com o blob removido da região válida
    """
    valid_px = gray[valid_region > 0]
    if valid_px.size < 200:
        return valid_region.copy()

    if polarity == "white_hot":
        thresh = max(float(np.percentile(valid_px, percentile)), 170.0)
        candidate = (gray >= thresh).astype(np.uint8) * 255
    else:  # black_hot
        thresh = min(float(np.percentile(valid_px, 100.0 - percentile)), 80.0)
        candidate = (gray <= thresh).astype(np.uint8) * 255

    # Restringir à região válida
    candidate = cv2.bitwise_and(candidate, valid_region)

    # Identificar o maior blob conectado (o alvo principal)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        candidate, connectivity=8
    )
    blob_mask = np.zeros_like(candidate)
    if n_labels > 1:  # label 0 é background
        areas = stats[1:, cv2.CC_STAT_AREA]
        if areas.max() >= min_blob_px:
            largest_label = int(np.argmax(areas)) + 1
            blob_mask[labels == largest_label] = 255

    # Dilatar para cobrir halo
    if np.count_nonzero(blob_mask) > 0 and dilation_px > 0:
        d = dilation_px * 2 + 1
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d, d))
        blob_mask = cv2.dilate(blob_mask, k)

    # Remover blob da região válida
    return cv2.bitwise_and(valid_region, cv2.bitwise_not(blob_mask))


def _mask_reticle(
    shape: tuple[int, int],
    center_frac: float = 0.12,
    line_frac: float = 0.015,
) -> np.ndarray:
    """
    Mascara a mira/crosshair central do sistema de armas.

    O retículo é fixo na imagem — não se move com o fundo — portanto
    contamina qualquer estimativa de movimento se incluído.
    Remove:
      - Banda horizontal (linha do crosshair)
      - Banda vertical (linha do crosshair)
      - Caixa central (onde o alvo e os marcadores ficam)

    Parâmetros
    ----------
    center_frac : fração das dimensões do frame para a caixa central
    line_frac   : espessura relativa das bandas horizontal/vertical

    Retorna
    -------
    Máscara uint8: 0 = retículo (excluído), 255 = fora do retículo
    """
    h, w = shape[:2]
    mask = np.ones((h, w), dtype=np.uint8) * 255
    cy, cx = h // 2, w // 2

    # Banda horizontal do crosshair
    lh = max(1, int(h * line_frac / 2))
    mask[max(0, cy - lh) : min(h, cy + lh + 1), :] = 0

    # Banda vertical do crosshair
    lw = max(1, int(w * line_frac / 2))
    mask[:, max(0, cx - lw) : min(w, cx + lw + 1)] = 0

    # Caixa central (alvo + marcadores do sistema de armas)
    hw = max(1, int(w * center_frac / 2))
    hh = max(1, int(h * center_frac / 2))
    mask[
        max(0, cy - hh) : min(h, cy + hh + 1),
        max(0, cx - hw) : min(w, cx + hw + 1),
    ] = 0

    return mask


def _build_background_mask(
    gray: np.ndarray,
    polarity: str = "white_hot",
    black_thresh: int = 18,
    center_frac: float = 0.12,
    line_frac: float = 0.015,
    blob_percentile: float = 99.0,
    blob_dilation_px: int = 28,
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
) -> tuple[np.ndarray, dict[str, float]]:
    """
    Constrói a máscara completa de background combinando todas as exclusões:

      1. Região válida (remove blocos pretos de HUD)
      2. Remoção do blob do alvo térmico (UAP)
      3. Remoção do retículo central
      4. Remoção de caixas HUD adicionais (texto, indicadores)

    Retorna (mask, info_dict) onde info_dict contém métricas de cobertura.
    """
    h, w = gray.shape[:2]

    # 1. Região válida (não-preta)
    valid = _find_valid_region(gray, black_thresh)
    valid_cov = float(np.count_nonzero(valid)) / valid.size

    # 2. Remover blob do alvo
    after_blob = _mask_target_blob(
        gray, valid, polarity, blob_percentile, blob_dilation_px
    )

    # 3. Remover retículo central
    reticle = _mask_reticle(gray.shape, center_frac, line_frac)
    mask = cv2.bitwise_and(after_blob, reticle)

    # 4. Caixas HUD adicionais informadas pelo usuário
    for (bx, by, bw, bh) in (hud_boxes or []):
        bx, by, bw, bh = int(bx), int(by), int(bw), int(bh)
        mask[max(0, by) : min(h, by + bh), max(0, bx) : min(w, bx + bw)] = 0

    bg_cov = float(np.count_nonzero(mask)) / mask.size

    return mask, {"valid_coverage": valid_cov, "bg_coverage": bg_cov}


print("Bloco 1 (máscara automática) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 2 — ESTIMATIVA DE MOVIMENTO POR FLUXO ÓPTICO ESPARSO
# ═══════════════════════════════════════════════════════════════════════════════

def _detect_features(
    gray: np.ndarray,
    mask: np.ndarray,
    max_corners: int = 400,
    quality: float = 0.004,
    min_dist: int = 8,
    block_size: int = 7,
) -> np.ndarray | None:
    """
    Detecta pontos de rastreamento no background usando o detector Shi-Tomasi.

    Shi-Tomasi (goodFeaturesToTrack) é preferido ao Harris neste contexto porque:
    - Funciona bem em texturas moderadas como nuvens e terreno
    - O parâmetro quality_level adapta o limiar ao conteúdo do frame
    - O parâmetro min_dist evita features agrupadas na mesma região

    A máscara garante que as features sejam detectadas APENAS na região de
    background (fora do alvo, fora do retículo, fora dos blocos pretos).
    """
    if np.count_nonzero(mask) < 300:
        return None

    pts = cv2.goodFeaturesToTrack(
        gray,
        maxCorners=max_corners,
        qualityLevel=quality,
        minDistance=min_dist,
        mask=mask,
        blockSize=block_size,
        useHarrisDetector=False,
    )
    return pts  # shape (N, 1, 2) ou None


def _track_features_lk(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    prev_pts: np.ndarray,
    win_size: int = 21,
    pyr_levels: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Rastreia pontos de frame para frame com Lucas-Kanade piramidal.

    A pirâmide (pyr_levels) é essencial para câmeras que fazem pans rápidos:
    os níveis superiores da pirâmide capturam grandes deslocamentos que seriam
    invisíveis no nível base. win_size controla o tamanho da janela de busca
    em cada nível — maior janela é mais robusta a ruído mas menos precisa.

    Retorna (pontos_rastreados, máscara_de_sucesso) onde máscara_de_sucesso
    é bool array com True para cada ponto rastreado com sucesso.
    """
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 25, 0.01)
    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        curr_gray,
        prev_pts,
        None,
        winSize=(win_size, win_size),
        maxLevel=pyr_levels,
        criteria=criteria,
    )
    tracked = status.ravel().astype(bool)
    return next_pts, tracked


def _robust_translation(
    prev_pts: np.ndarray,
    curr_pts: np.ndarray,
    tracked: np.ndarray,
    mad_k: float = 2.5,
    min_inliers: int = 5,
) -> tuple[float, float, int, int, float, bool]:
    """
    Extrai a translação dominante dos vetores de fluxo usando filtragem MAD.

    Vetores de fluxo idealmente devem apontar todos na mesma direção
    (movimento global da câmera). O alvo UAP, se vazar pela máscara, ou
    objetos secundários, produzem vetores outliers. A filtragem MAD
    (Median Absolute Deviation) remove esses outliers sem assumir
    distribuição gaussiana.

    Algoritmo:
      1. Compute vetor mediano (dx_med, dy_med)
      2. Compute MAD de cada componente separadamente
      3. Defina banda de inlier = k × MAD (mínimo 0.5 px para evitar rejeição excessiva)
      4. Recalcule mediana apenas com inliers

    Retorna (dx, dy, n_detectados, n_inliers, confiança_0_a_1, sucesso)
    """
    p = prev_pts[tracked].reshape(-1, 2)
    c = curr_pts[tracked].reshape(-1, 2)
    n_detected = len(p)

    if n_detected < min_inliers:
        return 0.0, 0.0, n_detected, 0, 0.0, False

    disp = c - p  # (N, 2)

    # Mediana robusta inicial
    dx_med = float(np.median(disp[:, 0]))
    dy_med = float(np.median(disp[:, 1]))

    # MAD por componente
    mad_x = float(np.median(np.abs(disp[:, 0] - dx_med)))
    mad_y = float(np.median(np.abs(disp[:, 1] - dy_med)))

    # Banda de inlier: pelo menos 0.5 px para evitar rejeição excessiva em
    # frames com pouco movimento (quando MAD → 0)
    thr_x = max(mad_k * mad_x, 0.5)
    thr_y = max(mad_k * mad_y, 0.5)

    inliers = (
        (np.abs(disp[:, 0] - dx_med) <= thr_x) &
        (np.abs(disp[:, 1] - dy_med) <= thr_y)
    )
    n_in = int(inliers.sum())

    if n_in >= min_inliers:
        dx_final = float(np.median(disp[inliers, 0]))
        dy_final = float(np.median(disp[inliers, 1]))
    else:
        dx_final, dy_final = dx_med, dy_med
        n_in = n_detected

    confidence = n_in / max(n_detected, 1)
    return dx_final, dy_final, n_detected, n_in, float(confidence), True


def _estimate_frame_motion(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    bg_mask: np.ndarray,
    max_corners: int = 400,
    win_size: int = 21,
    pyr_levels: int = 4,
    min_inliers: int = 5,
) -> dict[str, Any]:
    """
    Estima o deslocamento bg_dx, bg_dy entre dois frames consecutivos.

    Pipeline completo:
      Shi-Tomasi features (no background mascarado)
        → LK pyramidal tracking
          → MAD robust inlier filtering
            → translação dominante (mediana de inliers)

    Retorna
    -------
    dict com: bg_dx_px, bg_dy_px, n_features, n_inliers, confidence, motion_ok
    """
    result = {
        "bg_dx_px": 0.0, "bg_dy_px": 0.0,
        "n_features": 0, "n_inliers": 0,
        "confidence": 0.0, "motion_ok": False,
    }

    prev_pts = _detect_features(
        prev_gray, bg_mask,
        max_corners=max_corners,
    )
    if prev_pts is None or len(prev_pts) < min_inliers:
        result["n_features"] = 0 if prev_pts is None else len(prev_pts)
        return result

    curr_pts, tracked = _track_features_lk(
        prev_gray, curr_gray, prev_pts,
        win_size=win_size, pyr_levels=pyr_levels,
    )

    dx, dy, n_det, n_in, conf, ok = _robust_translation(
        prev_pts, curr_pts, tracked, min_inliers=min_inliers
    )

    result.update({
        "bg_dx_px": dx, "bg_dy_px": dy,
        "n_features": n_det, "n_inliers": n_in,
        "confidence": conf, "motion_ok": ok,
    })
    return result


print("Bloco 2 (fluxo óptico esparso LK) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 3 — PROCESSAMENTO DE SINAL: POSIÇÃO, VELOCIDADE E ACELERAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════

def _prefilter_displacements(
    dx: np.ndarray,
    dy: np.ndarray,
    ok: np.ndarray,
    window: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Pré-filtra os deslocamentos brutos frame-a-frame com mediana móvel centrada.

    Frames onde a estimativa falhou (ok=False) são marcados como NaN e
    interpolados pelo valor mediano dos vizinhos. Isso evita que frames
    problemáticos (cena totalmente desfocada, pan muito rápido, background
    quase liso) injetem saltos espúrios na integral de posição.

    O window deve ser ímpar para simetria. A mediana (vs média) mantém
    robustez a outliers que passaram pelo filtro MAD do LK.

    Parâmetros
    ----------
    dx, dy : deslocamentos brutos por frame (background)
    ok     : bool array — True onde a estimativa foi bem-sucedida
    window : janela da mediana móvel (frames)

    Retorna
    -------
    (dx_filtrado, dy_filtrado) como numpy arrays
    """
    w = window if window % 2 == 1 else window + 1
    w = max(w, 3)

    s_dx = pd.Series(np.where(ok, dx, np.nan))
    s_dy = pd.Series(np.where(ok, dy, np.nan))

    dx_filt = (
        s_dx.rolling(w, center=True, min_periods=1)
        .median()
        .fillna(0)
        .to_numpy(dtype=float)
    )
    dy_filt = (
        s_dy.rolling(w, center=True, min_periods=1)
        .median()
        .fillna(0)
        .to_numpy(dtype=float)
    )
    return dx_filt, dy_filt


def _compute_kinematics(
    position: np.ndarray,
    dt: float,
    window: int,
    poly: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Calcula posição suavizada, velocidade e aceleração via filtro Savitzky-Golay.

    O SG ajusta um polinômio local de grau `poly` a uma janela de `window`
    pontos. As derivadas são calculadas ANALITICAMENTE nesse polinômio —
    não numericamente — o que evita a amplificação de ruído que ocorre ao
    diferenciar numericamente depois de suavizar.

    Resultado:
      deriv=0 → posição suavizada
      deriv=1 → velocidade em px/s (divide pelo dt automaticamente)
      deriv=2 → aceleração em px/s² (divide por dt² automaticamente)

    Se scipy não estiver disponível, usa numpy.gradient (mais ruidoso).
    """
    if not _SCIPY or len(position) < window:
        smooth = position.copy()
        vel = np.gradient(smooth, dt)
        acc = np.gradient(vel, dt)
        return smooth, vel, acc

    smooth = savgol_filter(position, window, poly)
    vel    = savgol_filter(position, window, poly, deriv=1, delta=dt)
    acc    = savgol_filter(position, window, poly, deriv=2, delta=dt)
    return smooth, vel, acc


def _make_sg_window(n: int, window_s: float, eff_fps: float, poly: int) -> int:
    """
    Calcula tamanho de janela válido para Savitzky-Golay.

    Restrições:
      - Deve ser ímpar (requisito do SG)
      - Deve ser > poly (requisito do SG)
      - Deve ser <= número de amostras
    """
    w = max(poly + 2, int(round(window_s * eff_fps)))
    if w % 2 == 0:
        w += 1
    max_w = n if n % 2 != 0 else n - 1
    w = min(w, max(poly + 1, max_w))
    # Garantir paridade ímpar após clamp
    if w % 2 == 0:
        w -= 1
    return max(w, poly + 1 if (poly + 1) % 2 != 0 else poly + 2)


print("Bloco 3 (processamento de sinal) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 4 — FUNÇÃO PRINCIPAL
# ═══════════════════════════════════════════════════════════════════════════════

def _safe_fps(cap: cv2.VideoCapture, fallback: float = 30.0) -> float:
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    return fps if (math.isfinite(fps) and fps > 0) else fallback


def estimate_camera_motion(
    video_path: str | Path,
    *,
    # ── Detecção da região válida (blocos pretos) ──────────────────────────
    black_thresh: int = 18,
    # ── Polaridade térmica (white-hot vs black-hot) ────────────────────────
    target_polarity: str = "auto",
    # ── Mascaramento do blob do alvo (UAP) ────────────────────────────────
    blob_percentile: float = 99.0,
    blob_dilation_px: int = 28,
    # ── Mascaramento do retículo/mira ──────────────────────────────────────
    center_frac: float = 0.12,
    line_frac: float = 0.015,
    # ── Caixas HUD adicionais (texto, indicadores fixos) ───────────────────
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
    # ── Rastreamento de features (Shi-Tomasi + Lucas-Kanade) ───────────────
    max_features: int = 400,
    lk_win_size: int = 21,
    pyr_levels: int = 4,
    # ── Seleção de frames ──────────────────────────────────────────────────
    frame_step: int = 1,
    roi: tuple[int, int, int, int] | None = None,
    fps_override: float | None = None,
    # ── Pré-filtro de deslocamentos brutos ────────────────────────────────
    prefilter_window: int = 5,
    # ── Suavização Savitzky-Golay (posição → velocidade → aceleração) ──────
    smooth_window_s: float = 0.5,
    smooth_poly: int = 3,
    # ── Misc ───────────────────────────────────────────────────────────────
    verbose: bool = True,
) -> dict[str, Any]:
    """
    Estima posição, velocidade e aceleração da câmera FLIR de sistema de armas.

    A estimativa é feita EXCLUSIVAMENTE pelo movimento do background.
    Todos os elementos fixos ou de alto contraste (blocos pretos do sistema,
    o alvo UAP, o retículo, e texto HUD) são mascarados automaticamente
    antes de qualquer estimativa de movimento.

    Parâmetros de mascaramento
    --------------------------
    black_thresh
        Intensidade máxima para pixels serem considerados parte dos blocos
        pretos do sistema (típico: 18). Aumente se os blocos aparecerem
        parcialmente visíveis depois do processamento.
    target_polarity
        'auto' detecta automaticamente white-hot vs black-hot.
        Use 'white_hot' se o alvo é sempre brilhante, 'black_hot' se escuro.
    blob_percentile
        Percentil para threshold do blob (padrão 99 = mascara o 1% mais extremo).
        Reduza para 98 se o alvo é difuso/grande; aumente para 99.5 se compacto.
    blob_dilation_px
        Raio de dilatação do blob mascarado (cobre o halo térmico ao redor).
    center_frac
        Fração das dimensões do frame para a caixa central do retículo.
        Aumente se a mira/target box do sistema de armas for maior.
    hud_boxes
        Lista de (x, y, w, h) em pixels para excluir texto HUD fixo, como
        indicadores de altitude, velocidade, bússola, etc.

    Parâmetros de tracking (Lucas-Kanade)
    --------------------------------------
    max_features
        Máximo de features Shi-Tomasi por frame. Mais features = mais robusto
        mas mais lento. 200–600 é o intervalo prático.
    lk_win_size
        Tamanho da janela de busca LK por nível de pirâmide. Aumente (ex: 31)
        para pans rápidos onde o background desloca muito entre frames.
    pyr_levels
        Níveis da pirâmide LK. 3–4 é suficiente para a maioria dos vídeos.

    Parâmetros de sinal
    --------------------
    smooth_window_s
        Janela do filtro Savitzky-Golay em segundos. Controla o trade-off:
        janela maior → curvas mais suaves, menor resolução temporal das manobras;
        janela menor → detecta manobras rápidas, mais ruidoso.
    smooth_poly
        Grau do polinômio SG. Grau 3 é o padrão; use 2 para suavização mais forte.

    Retorna
    -------
    dict com:
        'metadata' : parâmetros usados e estatísticas
        'data'     : pd.DataFrame com colunas por frame

    Colunas principais do DataFrame
    --------------------------------
    frame_index, time_s
    bg_dx_px, bg_dy_px           : deslocamento bruto do fundo por frame
    bg_dx_filt_px, bg_dy_filt_px : deslocamento pré-filtrado (mediana móvel)
    cam_pos_x_px, cam_pos_y_px   : posição cumulativa da câmera (= -sum bg)
    cam_pos_x_smooth_px, [...]   : posição suavizada pelo SG
    cam_vel_x_px_s, cam_vel_y_px_s    : velocidade em px/s
    cam_acc_x_px_s2, cam_acc_y_px_s2  : aceleração em px/s²
    cam_speed_px_s                    : módulo da velocidade
    cam_accel_magnitude_px_s2         : módulo da aceleração
    n_features, n_inliers, confidence : qualidade da estimativa LK
    bg_coverage, valid_coverage       : cobertura da máscara
    motion_ok                         : True se estimativa foi bem-sucedida

    Convenção de sinal
    ------------------
    cam_pos_x > 0  →  câmera girou para a DIREITA (pan direita)
    cam_pos_y > 0  →  câmera inclinou para BAIXO (coordenadas de imagem)
    bg_dx_px > 0   →  fundo deslocou para a DIREITA (câmera foi para a esquerda)
    """
    video_path = Path(video_path).expanduser().resolve()
    if not video_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {video_path}")
    if frame_step < 1:
        raise ValueError("frame_step deve ser >= 1")
    if target_polarity not in {"auto", "white_hot", "black_hot"}:
        raise ValueError("target_polarity: 'auto' | 'white_hot' | 'black_hot'")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Não foi possível abrir o vídeo: {video_path}")

    fps      = fps_override or _safe_fps(cap)
    n_nom    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    vid_w    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)  or 0)
    vid_h    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    if verbose:
        print(f"Vídeo  : {video_path.name}")
        print(f"         {vid_w}×{vid_h} px  |  {fps:.3f} fps  |  ~{n_nom} frames")

    if roi is not None:
        rx, ry, rw, rh = (int(v) for v in roi)
        if not (0 <= rx and 0 <= ry and rw > 0 and rh > 0
                and rx + rw <= vid_w and ry + rh <= vid_h):
            raise ValueError(f"roi {roi} inválido para frame {vid_w}×{vid_h}")
    else:
        rx, ry, rw, rh = 0, 0, vid_w, vid_h

    dt      = frame_step / fps
    eff_fps = fps / frame_step

    rows: list[dict[str, Any]] = []
    prev_gray: np.ndarray | None = None
    polarity = target_polarity
    read_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        fi = read_idx
        read_idx += 1

        if fi % frame_step != 0:
            continue

        t_s  = fi / fps
        crop = frame[ry : ry + rh, rx : rx + rw]
        gray = _to_gray(crop)

        # No primeiro frame: inicializa e detecta polaridade
        if prev_gray is None:
            if target_polarity == "auto":
                valid0 = _find_valid_region(gray, black_thresh)
                polarity = _detect_thermal_polarity(gray, valid0)
                if verbose:
                    print(f"Polaridade detectada : {polarity}")
            rows.append({
                "frame_index": fi, "time_s": t_s,
                "bg_dx_px": 0.0, "bg_dy_px": 0.0,
                "n_features": 0, "n_inliers": 0,
                "confidence": 0.0,
                "bg_coverage": np.nan, "valid_coverage": np.nan,
                "motion_ok": False,
            })
            prev_gray = gray
            continue

        # Construir máscara de background
        bg_mask, cov = _build_background_mask(
            gray,
            polarity=polarity,
            black_thresh=black_thresh,
            center_frac=center_frac,
            line_frac=line_frac,
            blob_percentile=blob_percentile,
            blob_dilation_px=blob_dilation_px,
            hud_boxes=hud_boxes,
        )

        # Estimar movimento
        motion = _estimate_frame_motion(
            prev_gray, gray, bg_mask,
            max_corners=max_features,
            win_size=lk_win_size,
            pyr_levels=pyr_levels,
        )

        rows.append({
            "frame_index": fi, "time_s": t_s,
            **motion,
            **cov,
        })
        prev_gray = gray

    cap.release()

    if not rows:
        raise RuntimeError("Nenhum frame foi processado.")

    df = pd.DataFrame(rows)
    n  = len(df)

    # ── Pré-filtrar deslocamentos brutos ───────────────────────────────────
    dx_raw = df["bg_dx_px"].to_numpy(dtype=float)
    dy_raw = df["bg_dy_px"].to_numpy(dtype=float)
    ok_arr = df["motion_ok"].to_numpy(dtype=bool)

    dx_filt, dy_filt = _prefilter_displacements(
        dx_raw, dy_raw, ok_arr, prefilter_window
    )
    df["bg_dx_filt_px"] = dx_filt
    df["bg_dy_filt_px"] = dy_filt

    # ── Posição cumulativa da câmera ───────────────────────────────────────
    # Fundo desloca para a direita (+dx) → câmera girou para a esquerda → pos_x negativa
    # cam_pos = -cumsum(bg_deslocamento) → positivo = câmera girou para a direita
    df["cam_pos_x_px"] = -np.cumsum(dx_filt)
    df["cam_pos_y_px"] = -np.cumsum(dy_filt)

    # ── Savitzky-Golay: pos suave, velocidade, aceleração ─────────────────
    sg_w = _make_sg_window(n, smooth_window_s, eff_fps, smooth_poly)

    if verbose:
        n_ok = int(ok_arr.sum())
        print(f"\nFrames processados   : {n}")
        print(f"Janela SG            : {sg_w} frames ({sg_w * dt:.3f} s)")
        print(f"Estimativas OK       : {n_ok}/{max(n-1,1)} ({100*n_ok/max(n-1,1):.1f}%)")
        mc = df["bg_coverage"].dropna()
        if not mc.empty:
            print(f"Cobertura BG (média) : {mc.mean():.1%}")

    px = df["cam_pos_x_px"].to_numpy(dtype=float)
    py = df["cam_pos_y_px"].to_numpy(dtype=float)

    sx, vx, ax = _compute_kinematics(px, dt, sg_w, smooth_poly)
    sy, vy, ay = _compute_kinematics(py, dt, sg_w, smooth_poly)

    df["cam_pos_x_smooth_px"]      = sx
    df["cam_pos_y_smooth_px"]      = sy
    df["cam_vel_x_px_s"]           = vx
    df["cam_vel_y_px_s"]           = vy
    df["cam_acc_x_px_s2"]          = ax
    df["cam_acc_y_px_s2"]          = ay
    df["cam_speed_px_s"]           = np.hypot(vx, vy)
    df["cam_accel_magnitude_px_s2"]= np.hypot(ax, ay)

    metadata: dict[str, Any] = {
        "video_path"       : str(video_path),
        "fps"              : fps,
        "frame_step"       : frame_step,
        "eff_fps"          : eff_fps,
        "dt_s"             : dt,
        "video_size"       : (vid_w, vid_h),
        "roi"              : (rx, ry, rw, rh),
        "thermal_polarity" : polarity,
        "black_thresh"     : black_thresh,
        "blob_percentile"  : blob_percentile,
        "blob_dilation_px" : blob_dilation_px,
        "center_frac"      : center_frac,
        "line_frac"        : line_frac,
        "max_features"     : max_features,
        "lk_win_size"      : lk_win_size,
        "pyr_levels"       : pyr_levels,
        "smooth_window_s"  : smooth_window_s,
        "smooth_poly"      : smooth_poly,
        "sg_window_frames" : sg_w,
        "n_frames"         : n,
        "n_ok"             : int(ok_arr.sum()),
        "hud_boxes"        : hud_boxes or [],
    }

    return {"metadata": metadata, "data": df}


print("Bloco 4 (função principal) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 5 — VISUALIZAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════

def _shade_bad_frames(
    ax: plt.Axes,
    times: np.ndarray,
    ok: np.ndarray,
    alpha: float = 0.15,
) -> None:
    """
    Sombreia em vermelho claro os intervalos onde motion_ok=False.
    Ajuda a identificar regiões onde a estimativa foi ruim.
    """
    bad = ~ok.astype(bool)
    if not bad.any():
        return
    in_bad = False
    t_start = None
    for i, (t, b) in enumerate(zip(times, bad)):
        if b and not in_bad:
            t_start = t
            in_bad = True
        elif not b and in_bad:
            ax.axvspan(t_start, t, color="red", alpha=alpha, linewidth=0)
            in_bad = False
    if in_bad:
        ax.axvspan(t_start, times[-1], color="red", alpha=alpha, linewidth=0)


def plot_camera_kinematics(
    result: dict[str, Any],
    figsize: tuple[float, float] = (15, 11),
    title: str | None = None,
    shade_failures: bool = True,
) -> plt.Figure:
    """
    Gráfico principal: posição, velocidade e aceleração da câmera em X e Y.

    Layout: grade 3 × 2
    ┌─────────────────┬─────────────────┐
    │  Posição X (px) │  Posição Y (px) │  ← raw + suavizado SG
    ├─────────────────┼─────────────────┤
    │  Veloc. X (px/s)│  Veloc. Y (px/s)│  ← derivada SG
    ├─────────────────┼─────────────────┤
    │  Acel. X (px/s²)│  Acel. Y (px/s²)│  ← 2ª derivada SG
    └─────────────────┴─────────────────┘

    Regiões sombreadas em vermelho claro indicam frames onde a estimativa
    de movimento falhou (sem features suficientes no background).
    """
    df   = result["data"]
    meta = result["metadata"]
    t    = df["time_s"].to_numpy()
    ok   = df["motion_ok"].to_numpy()

    fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)
    _t = title or Path(meta["video_path"]).name
    fig.suptitle(
        f"Cinemática da Câmera FLIR\n{_t}",
        fontsize=13, fontweight="bold",
    )

    specs = [
        # (row, col, raw_col, smooth_col, vel_col, acc_col, color, axis_label)
        ("cam_pos_x_px",    "cam_pos_x_smooth_px", "cam_vel_x_px_s",
         "cam_acc_x_px_s2", "#2166ac", "X",
         "Pan   (+= direita)",  "Vel X", "Acel X"),
        ("cam_pos_y_px",    "cam_pos_y_smooth_px", "cam_vel_y_px_s",
         "cam_acc_y_px_s2", "#4dac26", "Y",
         "Tilt  (+= baixo)",   "Vel Y", "Acel Y"),
    ]

    vel_colors  = ["#d6604d", "#f4a582"]
    acc_colors  = ["#762a83", "#af8dc3"]

    for col_idx, (raw, smooth, vel, acc, color, ax_lbl,
                  pos_title, vel_title, acc_title) in enumerate(specs):

        # ── Posição ────────────────────────────────────────────────
        ax = axes[0, col_idx]
        ax.plot(t, df[raw],    color=color, alpha=0.2, lw=1,   label="raw")
        ax.plot(t, df[smooth], color=color, lw=2.2,            label="SG suavizado")
        ax.axhline(0, color="k", lw=0.7, alpha=0.3)
        ax.set_ylabel(f"Posição {ax_lbl} (px)", fontsize=10)
        ax.set_title(pos_title, fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)
        if shade_failures:
            _shade_bad_frames(ax, t, ok)

        # ── Velocidade ─────────────────────────────────────────────
        ax = axes[1, col_idx]
        ax.plot(t, df[vel], color=vel_colors[col_idx], lw=1.8)
        ax.axhline(0, color="k", lw=0.7, alpha=0.3)
        ax.set_ylabel(f"Velocidade {ax_lbl} (px/s)", fontsize=10)
        ax.set_title(vel_title, fontsize=9)
        ax.grid(True, alpha=0.25)
        if shade_failures:
            _shade_bad_frames(ax, t, ok)

        # ── Aceleração ─────────────────────────────────────────────
        ax = axes[2, col_idx]
        ax.plot(t, df[acc], color=acc_colors[col_idx], lw=1.8)
        ax.axhline(0, color="k", lw=0.7, alpha=0.3)
        ax.set_ylabel(f"Aceleração {ax_lbl} (px/s²)", fontsize=10)
        ax.set_xlabel("Tempo (s)", fontsize=10)
        ax.set_title(acc_title, fontsize=9)
        ax.grid(True, alpha=0.25)
        if shade_failures:
            _shade_bad_frames(ax, t, ok)

    # Rodapé com parâmetros
    sg_w = meta["sg_window_frames"]
    footer = (
        f"Polaridade: {meta['thermal_polarity']}  |  "
        f"Features: {meta['max_features']}  |  "
        f"LK win: {meta['lk_win_size']}  |  "
        f"SG: {sg_w} frames ({meta['smooth_window_s']:.2f}s, poly={meta['smooth_poly']})  |  "
        f"OK: {meta['n_ok']}/{meta['n_frames']-1}"
    )
    fig.text(0.5, 0.0, footer, ha="center", fontsize=7.5, color="#555555")
    fig.tight_layout(rect=[0, 0.02, 1, 1])
    return fig


def plot_camera_trajectory(
    result: dict[str, Any],
    figsize: tuple[float, float] = (7, 6),
) -> plt.Figure:
    """
    Trajetória 2D da câmera no plano da imagem, colorida por tempo.

    A cor vai de roxo escuro (início) a amarelo (fim), seguindo a paleta
    'plasma'. O ponto azul marca o início e o asterisco vermelho marca o fim.

    Pan a direita = deslocamento positivo no eixo X.
    Tilt para baixo = deslocamento positivo no eixo Y.
    """
    df   = result["data"]
    meta = result["metadata"]
    x    = df["cam_pos_x_smooth_px"].to_numpy()
    y    = df["cam_pos_y_smooth_px"].to_numpy()
    t    = df["time_s"].to_numpy()

    fig, ax = plt.subplots(figsize=figsize)

    pts  = np.stack([x, y], axis=1).reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc   = LineCollection(segs, cmap="plasma", linewidth=2.2,
                          norm=Normalize(t.min(), t.max()))
    lc.set_array(t[:-1])
    ax.add_collection(lc)
    cb = fig.colorbar(lc, ax=ax, pad=0.02)
    cb.set_label("Tempo (s)", fontsize=10)

    ax.scatter(x[0],  y[0],  c="royalblue", s=90, zorder=5,
               label="Início", marker="o")
    ax.scatter(x[-1], y[-1], c="crimson",   s=100, zorder=5,
               label="Fim",    marker="*")

    ax.set_xlabel("Câmera X (px)  [+ = pan direita]",  fontsize=10)
    ax.set_ylabel("Câmera Y (px)  [+ = tilt baixo]",   fontsize=10)
    ax.set_title(
        f"Trajetória 2D da Câmera\n{Path(meta['video_path']).name}",
        fontsize=11,
    )
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.autoscale()
    ax.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    return fig


def plot_motion_quality(
    result: dict[str, Any],
    figsize: tuple[float, float] = (14, 5),
) -> plt.Figure:
    """
    Métricas de qualidade do rastreamento frame-a-frame:
      - N.º de inliers (features aceitas pelo filtro MAD)
      - Confiança (n_inliers / n_detectados)
      - Cobertura do background (fração do frame usada)

    Valores baixos indicam frames onde o movimento é menos confiável.
    Use isso para ajustar blob_dilation_px, center_frac e hud_boxes.
    """
    df = result["data"]
    t  = df["time_s"].to_numpy()

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    fig.suptitle(
        "Qualidade da Estimativa de Movimento (LK Esparso)",
        fontsize=12, fontweight="bold",
    )

    ax = axes[0]
    ax.bar(t, df["n_inliers"], width=(t[1] - t[0]) * 0.9 if len(t) > 1 else 0.1,
           color="#4393c3", alpha=0.8, label="inliers")
    ax.bar(t, df["n_features"] - df["n_inliers"],
           bottom=df["n_inliers"],
           width=(t[1] - t[0]) * 0.9 if len(t) > 1 else 0.1,
           color="#d6604d", alpha=0.6, label="outliers (rejeitados)")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Nº de features")
    ax.set_title("Features detectadas vs. inliers")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25, axis="y")

    ax = axes[1]
    ax.plot(t, df["confidence"] * 100, color="#2ca02c", lw=1.5)
    ax.axhline(50, color="crimson", ls="--", lw=1, label="limiar 50%")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Confiança (%)")
    ax.set_title("Confiança (inliers / detectados)")
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    ax = axes[2]
    ax.plot(t, df["bg_coverage"] * 100, color="#8c510a", lw=1.5)
    ax.axhline(3, color="crimson", ls="--", lw=1, label="mínimo útil (3%)")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Cobertura BG (%)")
    ax.set_title("Fração do frame usada como background")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    fig.tight_layout()
    return fig


def visualize_mask(
    video_path: str | Path,
    frame_idx: int = 0,
    *,
    black_thresh: int = 18,
    target_polarity: str = "auto",
    blob_percentile: float = 99.0,
    blob_dilation_px: int = 28,
    center_frac: float = 0.12,
    line_frac: float = 0.015,
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
    roi: tuple[int, int, int, int] | None = None,
    figsize: tuple[float, float] = (14, 5),
) -> None:
    """
    Visualiza a máscara de background sobre um frame do vídeo.

    Painel esquerdo: frame original (cinza)
    Painel direito : frame com regiões EXCLUÍDAS em vermelho semitransparente

    Use esta função ANTES de rodar estimate_camera_motion() para confirmar
    que a máscara está correta e ajustar os parâmetros se necessário.

    Parâmetros
    ----------
    video_path  : caminho do vídeo
    frame_idx   : índice do frame a visualizar (0 = primeiro)
    Os demais parâmetros são os mesmos de estimate_camera_motion().
    """
    cap = cv2.VideoCapture(str(Path(video_path).expanduser().resolve()))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise ValueError(f"Não foi possível ler o frame {frame_idx}")

    if roi is not None:
        rx, ry, rw, rh = roi
        frame = frame[ry : ry + rh, rx : rx + rw]

    gray = _to_gray(frame)

    # Detectar polaridade se necessário
    polarity = target_polarity
    if polarity == "auto":
        v0 = _find_valid_region(gray, black_thresh)
        polarity = _detect_thermal_polarity(gray, v0)

    # Construir máscara
    mask, cov = _build_background_mask(
        gray, polarity=polarity,
        black_thresh=black_thresh,
        center_frac=center_frac, line_frac=line_frac,
        blob_percentile=blob_percentile,
        blob_dilation_px=blob_dilation_px,
        hud_boxes=hud_boxes,
    )

    # Detectar features no background
    pts = _detect_features(gray, mask, max_corners=300)

    # Montar visualização
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if frame.ndim == 3 else \
                cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)

    overlay = frame_rgb.copy().astype(np.float32)
    excluded = mask == 0
    overlay[excluded, 0] = np.clip(overlay[excluded, 0] * 0.3 + 200, 0, 255)
    overlay[excluded, 1] = overlay[excluded, 1] * 0.3
    overlay[excluded, 2] = overlay[excluded, 2] * 0.3
    overlay = overlay.astype(np.uint8)

    # Desenhar features
    if pts is not None:
        for pt in pts.reshape(-1, 2):
            cv2.circle(overlay, (int(pt[0]), int(pt[1])), 3, (255, 255, 0), -1)

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(frame_rgb, cmap="gray" if frame_rgb.ndim == 2 else None)
    axes[0].set_title(f"Frame original (índice {frame_idx})", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title(
        f"Máscara — vermelho=excluído, amarelo=features Shi-Tomasi ({len(pts) if pts is not None else 0} pts)\n"
        f"Polaridade: {polarity}  |  BG: {cov['bg_coverage']:.1%} do frame  |  "
        f"Válido: {cov['valid_coverage']:.1%}",
        fontsize=9,
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


print("Bloco 5 (visualização) carregado.")

## Como usar

### Passo 1 — Verificar a máscara antes de rodar (recomendado)

```python
visualize_mask(
    "meu_video.mp4",
    frame_idx=30,              # escolha um frame representativo
    blob_dilation_px=30,       # aumente se o halo do UAP vazar
    center_frac=0.15,          # aumente se a mira for grande
    hud_boxes=[(580, 10, 80, 25)],  # exemplo: exclui texto HUD no topo
)
```

### Passo 2 — Rodar a análise

```python
result = estimate_camera_motion(
    "meu_video.mp4",
    blob_dilation_px=30,
    center_frac=0.15,
    hud_boxes=[(580, 10, 80, 25)],
)
```

### Passo 3 — Plotar

```python
plot_camera_kinematics(result)   # posição, velocidade, aceleração
plot_camera_trajectory(result)   # trajetória 2D
plot_motion_quality(result)      # métricas de confiabilidade
```

---

### Guia de ajuste de parâmetros

| Sintoma | Diagnóstico | Parâmetro a ajustar |
|---|---|---|
| Cobertura BG < 3% | Máscara removeu quase tudo | Reduza `blob_dilation_px` ou `center_frac` |
| Muitos frames OK=False | Features insuficientes | Reduza `lk_win_size` ou `max_features ↑` |
| Saltos bruscos na posição | Outliers passando pelo MAD | Aumente `blob_dilation_px` |
| Curvas muito ruidosas | Suavização insuficiente | Aumente `smooth_window_s` |
| Manobras não aparecem | Suavização excessiva | Reduza `smooth_window_s` |
| Alvo escuro (black-hot) | Auto-detecção errada | Defina `target_polarity='black_hot'` |
| Blocos pretos não mascarados | Limiar muito baixo | Aumente `black_thresh` (ex: 25) |
| Pan muito rápido | LK perde rastreamento | Aumente `lk_win_size` (ex: 31) e `pyr_levels` |

### Interpretação dos gráficos

- **Posição**: deslocamento cumulativo da câmera desde o início do vídeo. Uma rampa constante = pan uniforme. Platô = câmera parada.
- **Velocidade**: taxa de pan/tilt instantânea. Picos = manobras. Deve ser zero quando câmera está estacionária.
- **Aceleração**: mudança de velocidade. Picos altos = manobras abruptas do gimbal para manter o UAP na mira.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  EXEMPLO DE USO
# ═══════════════════════════════════════════════════════════════════════════════

VIDEO = "/content/seu_video_flir.mp4"   # ← altere para o caminho do vídeo

# ── Passo 1: Verificar a máscara ─────────────────────────────────────────────
visualize_mask(
    VIDEO,
    frame_idx=30,
    black_thresh=18,          # pixels < 18 são considerados overlay preto
    target_polarity="auto",   # detecta white-hot / black-hot automaticamente
    blob_percentile=99.0,     # mascara o 1% mais extremo = o UAP
    blob_dilation_px=28,      # halo ao redor do blob mascarado
    center_frac=0.12,         # fração da mira central
    line_frac=0.015,          # espessura do crosshair
    hud_boxes=None,           # ex: [(x, y, w, h), ...] para texto HUD
)

In [ ]:
# ── Passo 2: Rodar estimativa de movimento ────────────────────────────────────
result = estimate_camera_motion(
    VIDEO,

    # Mascaramento
    black_thresh=18,
    target_polarity="auto",
    blob_percentile=99.0,
    blob_dilation_px=28,
    center_frac=0.12,
    line_frac=0.015,
    hud_boxes=None,

    # Rastreamento LK
    max_features=400,
    lk_win_size=21,
    pyr_levels=4,

    # Processamento de sinal
    prefilter_window=5,
    smooth_window_s=0.5,
    smooth_poly=3,

    frame_step=1,
    verbose=True,
)

# Resumo estatístico
df = result["data"]
print("\n── Estatísticas cinemáticas ──")
display(
    df[["cam_vel_x_px_s", "cam_vel_y_px_s",
        "cam_acc_x_px_s2", "cam_acc_y_px_s2",
        "cam_speed_px_s", "cam_accel_magnitude_px_s2"]]
    .describe()
    .round(2)
)

In [ ]:
# ── Passo 3: Gráficos ─────────────────────────────────────────────────────────

# Posição, velocidade e aceleração (X e Y)
fig1 = plot_camera_kinematics(result, shade_failures=True)
plt.show()

# Trajetória 2D colorida por tempo
fig2 = plot_camera_trajectory(result)
plt.show()

# Métricas de confiabilidade do rastreamento
fig3 = plot_motion_quality(result)
plt.show()

In [ ]:
# ── Exportar dados (opcional) ─────────────────────────────────────────────────
# result["data"].to_csv("camera_motion.csv", index=False)
# print("Exportado para camera_motion.csv")

# ── Inspecionar todas as colunas disponíveis ──────────────────────────────────
print("Colunas do DataFrame resultado:")
for col in result["data"].columns:
    sample = result["data"][col].iloc[1] if len(result["data"]) > 1 else None
    print(f"  {col:<35} ex: {sample}")